# Lesson 2 — Building Clean, Governed & Efficient Data Pipelines

**Enbridge Supply Chain Management — Databricks Enablement Workshop**

---

### What you'll learn in this lesson

| Section | Topic | Duration |
|---------|-------|---------|
| **A** | Data Governance & Quality using Unity Catalog | ~45 min |
| **B** | Building Silver & Gold Layers (Business Rules) | ~40 min |
| **C** | Performance and Cost-Aware Design | ~30 min |
| **D** | Building Pipelines with Notebooks & Lakeflow | ~40 min |
| **E** | Monitoring, Re-runs & Troubleshooting | ~20 min |
| **Bonus** | Compute Selection for Performance & Cost | ~10 min |

### Scenario
We will build a **Preferred Supplier List** data pipeline — the same data that today flows from
Oracle (ENBOAC schema) through Power BI. We'll replicate and improve that pipeline in Databricks
using the **medallion architecture** (Bronze → Silver → Gold), applying Enbridge's own business
rules (ISN Grade logic, BU parsing, geography formatting, approval status mapping).

### Source Data (in UC Volume)
```
/Volumes/<catalog>/<schema>/training/   (widget: volume_path)
├── Agreements - BPA.csv        (2,793 rows — Blanket Purchase Agreements)
├── Agreements - CPA.csv        (1,961 rows — Contract Purchase Agreements)
├── Contracts.csv               (1,858 rows — Legal contracts)
├── 12x Category Managed CSVs   (suppliers by BU & region)
├── 7x Regional Sourcing CSVs   (suppliers by region)
└── Preferred Supplier List.pbix (original Power BI report)
```

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Set your catalog and schema. Each participant can use their own schema.
dbutils.widgets.text("catalog", "serverless_stable_qr9if1_catalog", "1) Catalog")
dbutils.widgets.text("schema", "enbridge_scm", "2) Schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()

# Two source volumes:
# 1. training/  — original datasets (Agreements, Contracts, Category Managed CSVs, Regional Sourcing CSVs)
# 2. enbridge_psl_dashboard/ — enriched datasets (Supplier Headers, Contract Mapping, PO Line Items, CM Excel lists, etc.)
# SQL cells below read these widgets as :volume_path and :volume_path_psl
dbutils.widgets.text("volume_path", f"/Volumes/{CATALOG}/{SCHEMA}/training", "3) Volume - original CSVs")
dbutils.widgets.text("volume_path_psl", f"/Volumes/{CATALOG}/{SCHEMA}/enbridge_psl_dashboard", "4) Volume - PSL Dashboard files")
VOLUME_PATH = dbutils.widgets.get("volume_path").strip().rstrip("/")
VOLUME_PATH_PSL = dbutils.widgets.get("volume_path_psl").strip().rstrip("/")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")
print(f"Working in: {CATALOG}.{SCHEMA}")
print(f"Volume 1 (original):  {VOLUME_PATH}")
print(f"Volume 2 (enriched):   {VOLUME_PATH_PSL}")

---
# Section A — Data Governance & Quality Using Unity Catalog

> **Goal:** Structure your raw datasets in Unity Catalog with proper ownership, descriptions,
> and data quality standards. Learn the different table/view types and common ingestion patterns.

### Key Concepts
* **Unity Catalog 3-level namespace:** `catalog.schema.table`
* **Table types:** Managed tables, external tables, views, materialized views, streaming tables
* **Schema handling:** Schema inference, rescued data column, schema evolution
* **Governance:** Column comments, table descriptions, ownership, tags

## A.1 — Structuring Datasets: Schema Ownership & Documentation

Good governance starts with **organizing and documenting** your data assets.
Let's set up schema-level metadata and then ingest our raw supplier files.

In [0]:
%sql
-- Add a description to the schema so anyone can discover what it contains
COMMENT ON SCHEMA enbridge_scm IS 
  'Enbridge Supply Chain Management - Preferred Supplier List pipeline. Source: Oracle ENBOAC schema via Power BI migration.';

-- Verify
DESCRIBE SCHEMA EXTENDED enbridge_scm;

## A.2 — Bronze Layer: Ingesting Raw Data from Volumes

The **Bronze layer** holds raw data as-is from the source. We'll ingest from UC Volumes
using `read_files()` — a built-in function that handles schema inference automatically.

### Ingestion Patterns Covered
| Pattern | When to use | Used here |
|---------|------------|----------|
| `read_files()` (SQL) | Quick ingestion of files from Volumes | ✅ |
| Auto Loader (streaming) | Continuous/incremental file ingestion | Section D |
| COPY INTO | One-time bulk load with idempotency | Mentioned |
| `spark.read` (Python) | Programmatic file reads | Shown as alternative |

### Schema Handling Behaviors
* **Schema inference**: Databricks automatically detects column types from file contents
* **Rescued data column**: When rows don't match the inferred schema, mismatched data goes to `_rescued_data` instead of being lost
* **Header detection**: `header => 'true'` uses the first row as column names

In [0]:
%sql
-- ============================================================
-- BRONZE TABLE: Agreements - Blanket Purchase Agreements (BPA)
-- ============================================================
-- Note: The CPA file has duplicate "Report Date" columns. The rescued_data 
-- column automatically captures the second one — this is a real-world example
-- of how Databricks handles schema mismatches gracefully.

CREATE OR REPLACE TABLE bronze_agreements_bpa
COMMENT 'Raw Blanket Purchase Agreements from Oracle ENBOAC. Source: Power BI Preferred Supplier List.'
-- Column mapping lets Delta keep the original CSV headers (they contain spaces, e.g. `Supplier Number`).
-- Without it you get DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES. We rename to snake_case in Silver.
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS
SELECT 
  *,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM read_files(
  :volume_path || '/Agreements - BPA.csv',
  format => 'csv',
  header => 'true'
);

SELECT (SELECT count(*) FROM bronze_agreements_bpa) AS row_count, * FROM bronze_agreements_bpa LIMIT 5;

In [0]:
%sql
-- ============================================================
-- BRONZE TABLE: Agreements - Contract Purchase Agreements (CPA)  
-- ============================================================
-- This file has a duplicate "Report Date" column — observe how 
-- _rescued_data captures the second occurrence automatically.

CREATE OR REPLACE TABLE bronze_agreements_cpa
COMMENT 'Raw Contract Purchase Agreements from Oracle ENBOAC. Source: Power BI Preferred Supplier List.'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS
SELECT 
  *,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM read_files(
  :volume_path || '/Agreements - CPA.csv',
  format => 'csv',
  header => 'true'
);

-- Inspect the rescued data column to see what didn't fit the schema
SELECT `_rescued_data`, count(*) as cnt 
FROM bronze_agreements_cpa 
WHERE `_rescued_data` IS NOT NULL 
GROUP BY `_rescued_data` 
LIMIT 5;

In [0]:
%sql
-- ============================================================
-- BRONZE TABLE: Contracts
-- ============================================================
CREATE OR REPLACE TABLE bronze_contracts
COMMENT 'Raw legal contracts from Oracle ENBOAC. Includes contract terms, amounts, parties, and business units.'
TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
AS
SELECT 
  *,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM read_files(
  :volume_path || '/Contracts.csv',
  format => 'csv',
  header => 'true'
);

SELECT count(*) as row_count FROM bronze_contracts;

In [0]:
%sql
-- ============================================================
-- BRONZE TABLE: Category Managed Suppliers (all regions)
-- ============================================================
-- Ingesting 12 separate CSV files into ONE table using wildcard path.
-- This is a common pattern: land multiple source files as a single table.
-- The file name is captured as metadata so we can trace lineage.

CREATE OR REPLACE TABLE bronze_category_managed
COMMENT 'Preferred suppliers managed by category across all Enbridge business units (GTM, LP, GPT) and regions. Unioned from 12 source files.'
AS
SELECT 
  `Supplier Name`  AS supplier_name,
  `Supplier Number` AS supplier_number,
  `Business Unit`   AS business_unit,
  `Region`          AS region,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM read_files(
  :volume_path || '/*Category Managed*.csv',
  format => 'csv',
  header => 'true'
);

-- Verify: count by business unit and region
SELECT business_unit, region, count(*) as supplier_count 
FROM bronze_category_managed 
GROUP BY business_unit, region 
ORDER BY business_unit, region;

In [0]:
%sql
-- ============================================================
-- BRONZE TABLE: Regional Sourcing Suppliers (all regions)
-- ============================================================
CREATE OR REPLACE TABLE bronze_regional_sourcing
COMMENT 'Preferred suppliers for regional sourcing across all Enbridge business units and regions. Unioned from 7 source files.'
AS
SELECT 
  `Supplier Name`  AS supplier_name,
  `Supplier Number` AS supplier_number,
  `Business Unit`   AS business_unit,
  `Region`          AS region,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM read_files(
  :volume_path || '/*Regional Sourcing*.csv',
  format => 'csv',
  header => 'true'
);

SELECT business_unit, region, count(*) as supplier_count 
FROM bronze_regional_sourcing 
GROUP BY business_unit, region 
ORDER BY business_unit, region;

## A.6 — NEW: Ingesting Enriched Datasets from the PSL Dashboard Volume

The Enbridge team has provided an **enriched dataset** in a second volume (`enbridge_psl_dashboard/`).
These files complement our original data with richer supplier dimensions, contract mappings,
line-level spend data, and compliance tracking.

### New Datasets Overview

| File | Format | Rows | New Bronze Table | Purpose |
|------|--------|------|-----------------|--------|
| Supplier Header Info (2).csv | CSV | 29,909 | `bronze_supplier_headers` | Comprehensive supplier dimension (status, ISN#, registry ID) |
| *Contract Mapping*.csv (3 files) | CSV | ~2,203 | `bronze_contract_mapping` | Maps contracts to suppliers + L3 categories by region |
| Output1.xlsx | Excel | 50,350 | `bronze_po_line_items` | Line-level PO/contract financial data |
| Enbridge Standard Terms*.csv | CSV | 561 | `bronze_supplier_responses` | Supplier compliance/terms responses |
| Monthly Enterprise Cost Center*.xlsx | Excel | 2,881 | `bronze_cost_centers` | Cost center hierarchy dimension |
| ISN Province Region Mapping.xlsx | Excel | 444 | `bronze_isn_region_mapping` | ISN safety compliance reference |
| Region_Mapping_Updated*.xlsx | Excel | 288 | `bronze_region_mapping` | Ship-to location to region mapping |
| CM *.xlsx (8 files) | Excel | ~1,078 | `bronze_cm_supplier_lists` | New CM format with L3 Category + BU Yes/No flags |

> **Instructor note:** These datasets are ADDITIVE — we keep the original Bronze tables and
> build new ones alongside them. This demonstrates how to extend a pipeline when new data arrives.

In [0]:
%sql
-- ============================================================
-- NEW BRONZE: Supplier Header Info (CSV, ~30K rows)
-- ============================================================
-- Full supplier master from Oracle Cloud. Compare: the old workshop table had 24 suppliers.

CREATE OR REPLACE TABLE bronze_supplier_headers
COMMENT 'Supplier master from Oracle Cloud (~30K suppliers): status, relationship, ISN#, registry ID. Source: Supplier Header Info (2).csv'
AS SELECT
  `Supplier Name`                       AS supplier_name,
  CAST(`Supplier Number` AS STRING)     AS supplier_number,
  `Status`                              AS supplier_status,
  `Relationship`                        AS supplier_relationship,
  `ISN#`                                AS isn_number,
  `Registry ID`                         AS registry_id,
  _metadata.file_name                   AS source_file,
  current_timestamp()                   AS ingested_at
FROM read_files(
  :volume_path_psl || '/Supplier Header Info (2).csv',
  format => 'csv', header => true
);

SELECT supplier_status, count(*) AS suppliers FROM bronze_supplier_headers GROUP BY ALL ORDER BY suppliers DESC;

In [0]:
%sql
-- ============================================================
-- NEW BRONZE: Contract Mapping (3 regional CSVs via one wildcard)
-- ============================================================
CREATE OR REPLACE TABLE bronze_contract_mapping
COMMENT 'Contract-to-supplier mapping by BU, region and L3 category. Source: 3 regional Contract Mapping CSVs'
AS SELECT
  `Business Unit`                       AS business_unit,
  `Region`                              AS region,
  `Supplier Name`                       AS supplier_name,
  CAST(`Supplier Number` AS STRING)     AS supplier_number,
  CAST(`Contract Number` AS STRING)     AS contract_number,
  `L3 Category`                         AS l3_category,
  _metadata.file_name                   AS source_file,
  current_timestamp()                   AS ingested_at
FROM read_files(
  :volume_path_psl || '/*Contract Mapping*.csv',
  format => 'csv', header => true
);

SELECT source_file, count(*) AS rows FROM bronze_contract_mapping GROUP BY ALL;

In [0]:
%sql
-- ============================================================
-- NEW BRONZE: Standard Terms & Conditions - Supplier Responses (CSV)
-- ============================================================
CREATE OR REPLACE TABLE bronze_supplier_responses
COMMENT 'Supplier responses to Enbridge Standard Terms and Conditions. Source: Supplier Responses CSV'
AS SELECT
  CAST(`Supplier Number` AS STRING)     AS supplier_number,
  `Supplier Name`                       AS supplier_name,
  `ServiceNow Number`                   AS servicenow_number,
  `Question Name`                       AS question_name,
  `Response Value`                      AS response_value,
  CAST(`Response Date` AS TIMESTAMP)    AS response_date,
  _metadata.file_name                   AS source_file,
  current_timestamp()                   AS ingested_at
FROM read_files(
  :volume_path_psl || '/Enbridge Standard Terms and Conditions - Supplier Responses.csv',
  format => 'csv', header => true
);

SELECT response_value, count(*) AS responses FROM bronze_supplier_responses GROUP BY ALL;

### Ingesting Excel files with `read_files()`

`read_files()` reads `.xlsx` natively with `format => 'excel'` -- no pandas or extra libraries needed.
It reads the **first sheet** by default.

**Watch out:** Excel headers often contain spaces and symbols (`Supplier Name`, `Isnet by Prov/State`, `CONFIRM / CORRECT`).
Delta tables reject these characters, so a `SELECT *` will fail with `DELTA_INVALID_CHARACTERS_IN_COLUMN_NAMES`.
The fix used below: rename every column to snake_case in the Bronze `SELECT`.

> **Try it (optional):** change one of the next cells to `SELECT *` and run it to see the error.

In [0]:
%sql
-- ============================================================
-- NEW BRONZE: PO Line Items (Excel, ~50K rows)
-- ============================================================
-- Column names here are already safe (UPPER_SNAKE), so SELECT * works.
CREATE OR REPLACE TABLE bronze_po_line_items
COMMENT 'PO line items with ordered amounts by vendor (~50K rows). Source: Output1.xlsx'
AS SELECT
  *,
  _metadata.file_name AS source_file,
  current_timestamp() AS ingested_at
FROM read_files(
  :volume_path_psl || '/Output1.xlsx',
  format => 'excel', headerRows => 1
);

SELECT DOCUMENT_STATUS, count(*) AS lines FROM bronze_po_line_items GROUP BY ALL ORDER BY lines DESC;

In [0]:
%sql
-- ============================================================
-- NEW BRONZE: Category Managed supplier lists (8 Excel files, one wildcard)
-- ============================================================
-- New CM format: one row per supplier + L3 category, with 15 region Yes/No columns.
-- This complements (does not replace) bronze_category_managed from the original CSVs.
CREATE OR REPLACE TABLE bronze_cm_supplier_lists
COMMENT 'Category Managed supplier lists (new wide format): L3 category + 15 region Yes/No flags. Source: 8 CM Excel files'
AS SELECT
  CAST(CAST(`Oracle Cloud Supplier Number` AS BIGINT) AS STRING) AS supplier_number,
  TRIM(`Supplier Name`)               AS supplier_name,
  `L3 Category`                       AS l3_category,
  `LP Prairies`                       AS lp_prairies,
  `GTM Northwest`                     AS gtm_northwest,
  `LP Eastern`                        AS lp_eastern,
  `LP Athabasca`                      AS lp_athabasca,
  `LP Great Lakes`                    AS lp_great_lakes,
  `GTM Offshore & JV`                 AS gtm_offshore_jv,
  `LP US Gulf Coast Transmission`     AS lp_usgc_transmission,
  `LP Midwest`                        AS lp_midwest,
  `LP US Gulf Coast Terminal`         AS lp_usgc_terminal,
  `GTM Southeast`                     AS gtm_southeast,
  `GTM Central`                       AS gtm_central,
  `LP Southwest`                      AS lp_southwest,
  `GTM North Central`                 AS gtm_north_central,
  `GTM Northeast`                     AS gtm_northeast,
  `GPT Power Ops`                     AS gpt_power_ops,
  _metadata.file_name                 AS source_file,
  current_timestamp()                 AS ingested_at
FROM read_files(
  :volume_path_psl || '/*CM*.xlsx',
  format => 'excel', headerRows => 1
);

SELECT source_file, count(*) AS suppliers FROM bronze_cm_supplier_lists GROUP BY ALL ORDER BY suppliers DESC;

In [0]:
%sql
-- ============================================================
-- NEW BRONZE: Reference / mapping tables (3 Excel files)
-- ============================================================
CREATE OR REPLACE TABLE bronze_cost_centers
COMMENT 'Enterprise cost center hierarchy (Oct 2026). Source: Monthly Enterprise Cost Center Flat File'
AS SELECT
  `Enterprise Cost Center Code`        AS cost_center_code,
  `Parent`                             AS parent_code,
  `Level`                              AS level,
  `Description`                        AS description,
  `Business Unit`                      AS business_unit,
  `Dispatch Pool`                      AS dispatch_pool,
  `Regional Service Group (Dispatch)`  AS regional_service_group,
  `Overhead Group`                     AS overhead_group,
  `Region`                             AS region,
  `Area`                               AS area,
  `Report Month`                       AS report_month,
  `New vs Existing`                    AS new_vs_existing,
  _metadata.file_name                  AS source_file,
  current_timestamp()                  AS ingested_at
FROM read_files(
  :volume_path_psl || '/Monthly Enterprise Cost Center Flat File_October 2026.xlsx',
  format => 'excel', headerRows => 1
);

CREATE OR REPLACE TABLE bronze_isn_region_mapping
COMMENT 'ISN province/state applicability by region. Source: ISN Province Region Mapping.xlsx'
AS SELECT
  `Region`               AS region,
  `Isnet by Prov/State`  AS isnet_prov_state,
  `Applicable`           AS is_applicable,
  _metadata.file_name    AS source_file,
  current_timestamp()    AS ingested_at
FROM read_files(
  :volume_path_psl || '/ISN Province Region Mapping.xlsx',
  format => 'excel', headerRows => 1
);

CREATE OR REPLACE TABLE bronze_region_mapping
COMMENT 'Ship-to location to region mapping. Source: Region_Mapping_Updated_2026-10-19.xlsx (first sheet)'
AS SELECT
  `BUSINESS UNIT`          AS business_unit,
  `SHIP TO LOCATION NAME`  AS ship_to_location,
  `CURRENT DAX REGION`     AS current_region,
  `PROPOSED REGION`        AS proposed_region,
  `CONFIRM / CORRECT`      AS confirm_correct,
  `COMMENTS`               AS comments,
  _metadata.file_name      AS source_file,
  current_timestamp()      AS ingested_at
FROM read_files(
  :volume_path_psl || '/Region_Mapping_Updated_2026-10-19.xlsx',
  format => 'excel', headerRows => 1
);

SELECT 'bronze_cost_centers' AS table_name, count(*) AS rows FROM bronze_cost_centers
UNION ALL SELECT 'bronze_isn_region_mapping', count(*) FROM bronze_isn_region_mapping
UNION ALL SELECT 'bronze_region_mapping', count(*) FROM bronze_region_mapping;

## A.3 — Table and View Types in Databricks

Databricks supports several storage objects. Understanding **when to use each** is critical:

| Type | Persisted? | Auto-refresh? | Best for |
|------|-----------|--------------|----------|
| **Managed Table** | ✅ Data + metadata in UC | No | Bronze/Silver/Gold layers, ETL output |
| **External Table** | ✅ Data in your storage | No | Data shared across platforms |
| **View** | ❌ Logical only | N/A (computed on read) | Reusable query logic, row-level security |
| **Materialized View** | ✅ Pre-computed results | ✅ In Lakeflow pipelines | Gold aggregations, dashboard acceleration |
| **Streaming Table** | ✅ Append-only | ✅ In Lakeflow pipelines | Real-time/incremental ingestion |

> **💡 Tip:** In the Preferred Supplier List pipeline, we use **managed tables** for Bronze/Silver/Gold
> layers (simple, governed, performant). In Section D, we'll convert this to a **Lakeflow pipeline**
> using streaming tables and materialized views.

In [0]:
%sql
-- Let's see all our Bronze tables and understand table types

-- A VIEW is a saved query — it doesn't store data, just the SQL logic.
-- Great for reusable filters or security policies.
CREATE OR REPLACE VIEW vw_active_contracts AS
SELECT * FROM bronze_contracts
WHERE `Contract Status` = 'Active';

-- Check: A view has no storage cost, it computes on every read
SELECT count(*) as active_contracts FROM vw_active_contracts;

-- Our managed tables (bronze_*) store actual data in Unity Catalog
SHOW TABLES IN enbridge_scm;

## A.4 — Adding Governance: Column Comments, Tags & Constraints

Proper documentation makes data **discoverable** and **trustworthy**. Let's add:
* Column-level comments (business definitions)
* Table-level tags (for classification and search)
* NOT NULL constraints (for data quality enforcement)

In [0]:
%sql
-- Add business-meaningful descriptions to key columns
ALTER TABLE bronze_contracts ALTER COLUMN `Contract Number`  COMMENT 'Unique identifier for the contract in Oracle Cloud';
ALTER TABLE bronze_contracts ALTER COLUMN `Contract Status`  COMMENT 'Current status: Active, Expired, Terminated, etc.';
ALTER TABLE bronze_contracts ALTER COLUMN `Contract Amount`  COMMENT 'Total contract value in the header currency';
ALTER TABLE bronze_contracts ALTER COLUMN `Enbridge Business Unit` COMMENT 'BU code: LP (Liquids Pipelines), GTM (Gas Transmission), GDS (Enbridge Gas), GPT (Power Operations)';
ALTER TABLE bronze_contracts ALTER COLUMN `Contract Party Name`   COMMENT 'Legal name of the contracting supplier/vendor';

-- Add tags for classification (helps with search and governance policies)
-- Note: if your workspace has governed tag policies, some keys/values may be restricted
-- (error UC_TAG_POLICY_VALUE_NOT_ALLOWED). Pick a key that is not governed, or an allowed value.
ALTER TABLE bronze_contracts SET TAGS ('domain' = 'supply_chain', 'layer' = 'bronze', 'scm_source' = 'oracle_enboac');
ALTER TABLE bronze_category_managed SET TAGS ('domain' = 'supply_chain', 'layer' = 'bronze', 'scm_source' = 'oracle_enboac');
ALTER TABLE bronze_agreements_bpa SET TAGS ('domain' = 'supply_chain', 'layer' = 'bronze', 'scm_source' = 'oracle_enboac');
ALTER TABLE bronze_agreements_cpa SET TAGS ('domain' = 'supply_chain', 'layer' = 'bronze', 'scm_source' = 'oracle_enboac');

-- Verify the metadata
DESCRIBE TABLE EXTENDED bronze_contracts;

## A.5 — Data Quality Patterns: Validation, Rescued Data & Quarantine

### Key Data Quality Patterns

| Pattern | Purpose | Example |
|---------|---------|--------|
| **Rescued Data Column** | Capture schema-mismatched values instead of losing them | CPA file's duplicate "Report Date" column |
| **Validation Checks** | Assert expected values before promoting to Silver | Check for NULL supplier numbers, valid status codes |
| **Consistency Rules** | Enforce business constraints | Agreement dates must be valid, BU codes must be known |
| **Quarantine Pattern** | Isolate bad records for review | Route failed validations to a separate table |
| **Traceability** | Track data lineage | `source_file`, `ingested_at` columns on every table |

In [0]:
%sql
-- ============================================================
-- DATA QUALITY CHECK: Inspect the rescued data from CPA ingestion
-- ============================================================
-- The CPA file has TWO "Report Date" columns. The second one was
-- automatically rescued. Let's examine it.

SELECT 
  `Report Date`,
  `_rescued_data`,
  `Agreement Number`,
  `Supplier`
FROM bronze_agreements_cpa 
WHERE `_rescued_data` IS NOT NULL 
LIMIT 10;

In [0]:
%sql
-- ============================================================
-- QUARANTINE PATTERN: Separate good records from bad ones
-- ============================================================
-- This is a common Silver-layer pattern: validate before promoting.

-- Step 1: Identify quality issues in bronze_category_managed
SELECT 
  'Missing supplier_number' AS issue,
  count(*) AS affected_rows
FROM bronze_category_managed 
WHERE supplier_number IS NULL

UNION ALL

SELECT 
  'Unknown business_unit' AS issue,
  count(*) AS affected_rows
FROM bronze_category_managed 
WHERE business_unit NOT IN ('LP', 'GTM', 'GPT', 'GDS')

UNION ALL

SELECT 
  'Missing supplier_name' AS issue,
  count(*) AS affected_rows
FROM bronze_category_managed 
WHERE supplier_name IS NULL OR TRIM(supplier_name) = '';

-- 💡 In a production pipeline, rows failing these checks would be routed
-- to a quarantine table (bronze_category_managed_quarantine) for review,
-- while passing rows proceed to Silver.

---
# Section B — Building Silver & Gold Layers (Applying Enbridge Business Rules)

> **Goal:** Transform raw Bronze data into clean, business-ready tables using the actual
> transformation logic from the Power BI → Databricks SQL documentation.

### Enbridge Business Rules (from SQL Source & Transformation Documentation)

| Rule | Description | Used in |
|------|------------|--------|
| **R1** | ISN Grade: expired→NULL, blocking→'N/A - Contractor Blocking', else grade | Suppliers, Category Search |
| **R2** | BU parsing: extract business unit code from sourcing rule name | Category Search, Approved Categories |
| **R3** | Geography: format as 'Parent Region - Region' or fallback to lookup | Category Search, Approved Categories |
| **R4** | BU code→full name: LP→'Liquids Pipelines', GTM→'Gas Transmission', etc. | Category Search (ISN join) |
| **R5** | Approval Status: rank 1→Approved, 2→In Process, 3→Debarred, etc. | Category Search, Approved Categories |

## B.1 — Silver Layer: Unified Preferred Suppliers

Combine **Category Managed** and **Regional Sourcing** supplier lists into a single,
cleaned Silver table. Apply business rule **R4** (BU code to full name mapping).

In [0]:
%sql
-- ============================================================
-- SILVER TABLE: Unified Preferred Suppliers
-- ============================================================
-- Combines Category Managed + Regional Sourcing into one table.
-- Applies Business Rule R4: BU code → full business unit name.

CREATE OR REPLACE TABLE silver_preferred_suppliers
COMMENT 'Unified preferred supplier list with BU name resolution. Silver layer — cleaned and standardized.'
AS
SELECT 
  supplier_name,
  CAST(supplier_number AS INT) AS supplier_number,
  business_unit AS bu_code,
  -- Business Rule R4: Map BU code to full name
  CASE business_unit
    WHEN 'LP'  THEN 'Liquids Pipelines'
    WHEN 'GTM' THEN 'Gas Transmission'
    WHEN 'GDS' THEN 'Enbridge Gas'
    WHEN 'GPT' THEN 'Power Operations'
    ELSE business_unit
  END AS business_unit_name,
  region,
  'Category Managed' AS supplier_type,
  source_file,
  ingested_at
FROM bronze_category_managed

UNION ALL

SELECT 
  supplier_name,
  CAST(supplier_number AS INT) AS supplier_number,
  business_unit AS bu_code,
  CASE business_unit
    WHEN 'LP'  THEN 'Liquids Pipelines'
    WHEN 'GTM' THEN 'Gas Transmission'
    WHEN 'GDS' THEN 'Enbridge Gas'
    WHEN 'GPT' THEN 'Power Operations'
    ELSE business_unit
  END AS business_unit_name,
  region,
  'Regional Sourcing' AS supplier_type,
  source_file,
  ingested_at
FROM bronze_regional_sourcing;

-- Add governance metadata
ALTER TABLE silver_preferred_suppliers SET TAGS ('domain' = 'supply_chain', 'layer' = 'silver');
ALTER TABLE silver_preferred_suppliers ALTER COLUMN bu_code COMMENT 'Business unit code: LP, GTM, GDS, GPT';
ALTER TABLE silver_preferred_suppliers ALTER COLUMN business_unit_name COMMENT 'Full BU name per Rule R4: LP=Liquids Pipelines, GTM=Gas Transmission, GDS=Enbridge Gas, GPT=Power Operations';

SELECT * FROM silver_preferred_suppliers LIMIT 10;

## B.2 — Silver Layer: Cleaned Agreements

Standardize both agreement types (BPA and CPA) into a unified Silver table.
Handle the rescued data from the duplicate "Report Date" column in CPA.

In [0]:
%sql
-- ============================================================
-- SILVER TABLE: Unified Agreements (BPA + CPA)
-- ============================================================
-- Handles the duplicate Report Date column in CPA via rescued data.
-- Standardizes both agreement types into a single schema.

CREATE OR REPLACE TABLE silver_agreements
COMMENT 'Unified BPA and CPA agreements. Silver layer with standardized dates and types.'
AS
WITH bpa_cleaned AS (
  SELECT 
    `Document Type` AS document_type,
    TRY_CAST(`Supplier Number` AS INT) AS supplier_number,
    `Supplier` AS supplier_name,
    `Site` AS site,
    `Inactive Date` AS inactive_date,
    `Contract Number` AS contract_number,
    CAST(`Agreement Number` AS STRING) AS agreement_number,  -- BPA infers STRING, CPA infers BIGINT: align types before UNION
    `Agreement Status Lookup Code` AS agreement_status,
    `Report Date` AS report_start_date,
    -- BPA has a proper second Report Date column: "Requisition Business Unit Name"
    -- which is distinct from CPA's duplicate
    `Requisition Business Unit Name` AS requisition_bu_name,
    `Preferred Status` AS preferred_status,
    `Maximo Region` AS maximo_region,
    `Enbridge Regional Area` AS enbridge_regional_area,
    `Category Name` AS category_name,
    source_file,
    ingested_at
  FROM bronze_agreements_bpa
),
cpa_cleaned AS (
  SELECT 
    `Document Type` AS document_type,
    TRY_CAST(`Supplier Number` AS INT) AS supplier_number,
    `Supplier` AS supplier_name,
    `Site` AS site,
    `Inactive Date` AS inactive_date,
    `Contract Number` AS contract_number,
    CAST(`Agreement Number` AS STRING) AS agreement_number,  -- BPA infers STRING, CPA infers BIGINT: align types before UNION
    `Agreement Status Lookup Code` AS agreement_status,
    `Report Date` AS report_start_date,
    -- Extract the second Report Date from rescued data (it's the end date)
    TRY_CAST(get_json_object(`_rescued_data`, '$.Report Date_9') AS DATE) AS report_end_date,  -- TRY_CAST: blanks become NULL instead of failing
    `Preferred Status` AS preferred_status,
    `Maximo Region` AS maximo_region,
    `Enbridge Regional Area` AS enbridge_regional_area,
    source_file,
    ingested_at
  FROM bronze_agreements_cpa
)
SELECT 
  document_type, supplier_number, supplier_name, site, inactive_date,
  contract_number, agreement_number, agreement_status,
  report_start_date, NULL AS report_end_date,
  preferred_status, maximo_region, enbridge_regional_area,
  requisition_bu_name, category_name,
  source_file, ingested_at
FROM bpa_cleaned

UNION ALL

SELECT 
  document_type, supplier_number, supplier_name, site, inactive_date,
  contract_number, agreement_number, agreement_status,
  report_start_date, report_end_date,
  preferred_status, maximo_region, enbridge_regional_area,
  NULL AS requisition_bu_name, NULL AS category_name,
  source_file, ingested_at
FROM cpa_cleaned;

ALTER TABLE silver_agreements SET TAGS ('domain' = 'supply_chain', 'layer' = 'silver');

-- Quick look at the unified result
SELECT document_type, agreement_status, count(*) as cnt
FROM silver_agreements
GROUP BY document_type, agreement_status
ORDER BY document_type, cnt DESC;

## B.3 — NEW: Silver Layer for Enriched Datasets

Transform the new Bronze tables into clean, business-ready Silver tables:
* **`silver_supplier_enriched`** — Joins supplier headers with CM supplier lists (adds L3 category, ISN#, status)
* **`silver_contract_mapping`** — Cleaned contract mapping with Business Rule R4 applied
* **`silver_po_spend`** — PO line items with cleaned amounts and BU extraction

In [0]:
%sql
-- ============================================================
-- NEW SILVER: Supplier master enriched with Category Managed info
-- ============================================================
-- LEFT JOIN keeps every supplier; CM suppliers get their L3 category and
-- the number of regions they are approved for (count of 'Yes' flags).
CREATE OR REPLACE TABLE silver_supplier_enriched
COMMENT 'Supplier master enriched with CM L3 category and number of approved regions. One row per supplier x L3 category. Silver layer.'
AS SELECT
  h.supplier_number,
  h.supplier_name,
  h.supplier_status,
  h.supplier_relationship,
  h.isn_number,
  h.registry_id,
  c.l3_category,
  ( CASE WHEN c.lp_prairies          = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gtm_northwest        = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_eastern           = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_athabasca         = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_great_lakes       = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gtm_offshore_jv      = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_usgc_transmission = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_midwest           = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_usgc_terminal     = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gtm_southeast        = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gtm_central          = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.lp_southwest         = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gtm_north_central    = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gtm_northeast        = 'Yes' THEN 1 ELSE 0 END
  + CASE WHEN c.gpt_power_ops        = 'Yes' THEN 1 ELSE 0 END ) AS num_cm_regions,
  c.supplier_number IS NOT NULL AS is_category_managed
FROM bronze_supplier_headers h
LEFT JOIN bronze_cm_supplier_lists c
  ON h.supplier_number = c.supplier_number;

SELECT is_category_managed, count(DISTINCT supplier_number) AS suppliers FROM silver_supplier_enriched GROUP BY ALL;

In [0]:
%sql
-- ============================================================
-- NEW SILVER: Contract mapping with Business Rule R4 (BU code -> full name)
-- ============================================================
CREATE OR REPLACE TABLE silver_contract_mapping
COMMENT 'Contract-to-supplier mapping with BU full name (Rule R4) and L3 category. Silver layer.'
AS SELECT
  business_unit AS bu_code,
  CASE business_unit
    WHEN 'LP'  THEN 'Liquids Pipelines'
    WHEN 'GTM' THEN 'Gas Transmission'
    WHEN 'GDS' THEN 'Enbridge Gas'
    WHEN 'GPT' THEN 'Power Operations'
    ELSE business_unit
  END AS business_unit_name,
  region,
  supplier_name,
  supplier_number,
  contract_number,
  l3_category,
  source_file
FROM bronze_contract_mapping
WHERE supplier_number IS NOT NULL;

SELECT business_unit_name, region, count(DISTINCT supplier_number) AS suppliers
FROM silver_contract_mapping GROUP BY ALL ORDER BY suppliers DESC;

In [0]:
%sql
-- ============================================================
-- NEW SILVER: PO line items with proper types
-- ============================================================
-- Excel loads everything as STRING. Silver is where we type it:
-- try_to_date / TRY_CAST return NULL instead of failing on bad values.
CREATE OR REPLACE TABLE silver_po_spend
COMMENT 'PO line items with typed amounts and dates. Silver layer.'
AS SELECT
  REQ_BU_NAME                                        AS requisition_bu_name,
  DOCUMENT_NUM                                       AS document_number,
  DOCUMENT_STATUS                                    AS document_status,
  try_to_date(CREATION_DATE, 'MM/dd/yyyy')           AS creation_date,
  AGENT_NAME                                         AS agent_name,
  TRIM(VENDOR_NAME)                                  AS vendor_name,
  CONTRACT_DFF                                       AS contract_reference,
  TRY_CAST(LINE_NUM AS INT)                          AS line_number,
  TRY_CAST(TOTAL_ORDERED_AMOUNT AS DECIMAL(18,2))    AS total_ordered_amount,
  TEMPLATE_NAME                                      AS template_name
FROM bronze_po_line_items;

SELECT document_status, count(*) AS lines, sum(total_ordered_amount) AS total_amount
FROM silver_po_spend GROUP BY ALL ORDER BY lines DESC;

In [0]:
%sql
-- ============================================================
-- NEW GOLD: Supplier spend summary
-- ============================================================
CREATE OR REPLACE TABLE gold_supplier_spend_summary
COMMENT 'PO spend by vendor and document status. Gold layer for spend dashboards.'
AS SELECT
  vendor_name                      AS supplier_name,
  document_status,
  count(*)                         AS num_line_items,
  count(DISTINCT document_number)  AS num_documents,
  sum(total_ordered_amount)        AS total_spend,
  avg(total_ordered_amount)        AS avg_line_amount,
  max(total_ordered_amount)        AS max_line_amount
FROM silver_po_spend
WHERE vendor_name IS NOT NULL
GROUP BY vendor_name, document_status;

SELECT supplier_name, sum(total_spend) AS total_spend
FROM gold_supplier_spend_summary GROUP BY ALL ORDER BY total_spend DESC LIMIT 10;

In [0]:
%sql
-- ============================================================
-- NEW GOLD: L3 category coverage by BU and region (with liquid clustering)
-- ============================================================
CREATE OR REPLACE TABLE gold_category_coverage
CLUSTER BY (bu_code, region)
COMMENT 'Supplier and contract coverage per L3 category by BU and region. Gold layer for coverage-gap analysis.'
AS SELECT
  bu_code,
  business_unit_name,
  region,
  l3_category,
  count(DISTINCT supplier_number)  AS unique_suppliers,
  count(DISTINCT contract_number)  AS linked_contracts
FROM silver_contract_mapping
WHERE l3_category IS NOT NULL
GROUP BY bu_code, business_unit_name, region, l3_category;

-- Coverage gaps: categories with only one supplier in a region (single-source risk)
SELECT region, l3_category, unique_suppliers
FROM gold_category_coverage
WHERE unique_suppliers = 1
ORDER BY region, l3_category;

In [0]:
%sql
-- ============================================================
-- NEW GOLD: Supplier compliance with Standard Terms & Conditions
-- ============================================================
CREATE OR REPLACE TABLE gold_supplier_compliance
COMMENT 'Supplier compliance with Enbridge Standard Terms and Conditions. Gold layer.'
AS SELECT
  supplier_name,
  supplier_number,
  question_name,
  response_value,
  response_date,
  CASE
    WHEN response_value = 'Yes' THEN 'Compliant'
    WHEN response_value = 'No'  THEN 'Non-Compliant'
    ELSE 'Pending'
  END AS compliance_status
FROM bronze_supplier_responses
WHERE supplier_number IS NOT NULL;

SELECT compliance_status, count(*) AS suppliers FROM gold_supplier_compliance GROUP BY ALL;

## B.3 — Gold Layer: Business-Ready Analytics Tables

Gold tables are **optimized for consumption** — dashboards, Genie, reports.
We build two Gold tables that mirror what the Power BI report needs.

In [0]:
%sql
-- ============================================================
-- GOLD TABLE: Regional Supplier Coverage
-- ============================================================
-- One row per BU + Region with supplier counts and type breakdown.
-- Powers the regional coverage dashboard.

CREATE OR REPLACE TABLE gold_regional_supplier_coverage
COMMENT 'Regional supplier coverage summary by business unit and region. Gold layer for dashboards and reporting.'
AS
SELECT 
  bu_code,
  business_unit_name,
  region,
  supplier_type,
  COUNT(DISTINCT supplier_number) AS unique_suppliers,
  COUNT(*) AS total_assignments
FROM silver_preferred_suppliers
GROUP BY bu_code, business_unit_name, region, supplier_type
ORDER BY business_unit_name, region;

ALTER TABLE gold_regional_supplier_coverage SET TAGS ('domain' = 'supply_chain', 'layer' = 'gold');

SELECT * FROM gold_regional_supplier_coverage;

In [0]:
%sql
-- ============================================================
-- GOLD TABLE: Agreement Status Summary  
-- ============================================================
-- Aggregated view of all agreements for executive dashboards.

CREATE OR REPLACE TABLE gold_agreement_summary
COMMENT 'Aggregated agreement status summary by type and supplier. Gold layer for executive dashboards.'
AS
SELECT 
  document_type,
  agreement_status,
  COUNT(DISTINCT supplier_number) AS unique_suppliers,
  COUNT(DISTINCT agreement_number) AS total_agreements,
  COUNT(DISTINCT contract_number) AS total_contracts,
  MIN(report_start_date) AS earliest_report_date,
  MAX(report_start_date) AS latest_report_date
FROM silver_agreements
GROUP BY document_type, agreement_status
ORDER BY document_type, total_agreements DESC;

ALTER TABLE gold_agreement_summary SET TAGS ('domain' = 'supply_chain', 'layer' = 'gold');

SELECT * FROM gold_agreement_summary;

In [0]:
%sql
-- ============================================================
-- GOLD TABLE: Contract Analytics by Business Unit
-- ============================================================
CREATE OR REPLACE TABLE gold_contract_analytics
COMMENT 'Contract analytics by business unit, status, and type. Gold layer for spend analysis.'
AS
SELECT 
  `Enbridge Business Unit` AS business_unit,
  `Contract Status` AS contract_status,
  `Contract Type` AS contract_type,
  `Contract Classification` AS contract_classification,
  COUNT(*) AS contract_count,
  COUNT(DISTINCT `Contract Party Name`) AS unique_suppliers,
  MIN(`Contract Start Date`) AS earliest_start,
  MAX(`Contract End Date`) AS latest_end
FROM bronze_contracts
WHERE `Enbridge Business Unit` IS NOT NULL
GROUP BY `Enbridge Business Unit`, `Contract Status`, `Contract Type`, `Contract Classification`
ORDER BY business_unit, contract_count DESC;

ALTER TABLE gold_contract_analytics SET TAGS ('domain' = 'supply_chain', 'layer' = 'gold');

SELECT * FROM gold_contract_analytics;

## B.4 — NEW: Gold Tables for Enriched Analytics

Three new Gold tables built from the enriched datasets:
* **`gold_supplier_spend_summary`** — Total PO spend by vendor (from 50K line items)
* **`gold_category_coverage`** — L3 category coverage by BU and region
* **`gold_supplier_compliance`** — Supplier terms compliance status

---
# Section C — Performance and Cost-Aware Design

> **Goal:** Learn techniques for efficient pipeline design that minimize cost and improve performance.

### Key Techniques
* **Liquid Clustering** — automatically organizes data for fast queries (replaces Z-ORDER)
* **Optimized joins** — broadcast small tables, push predicates down
* **Reuse intermediate results** — temp views and CTEs avoid redundant processing
* **Column pruning** — SELECT only what you need

In [0]:
%sql
-- ============================================================
-- PERFORMANCE: Liquid Clustering
-- ============================================================
-- Liquid Clustering replaces PARTITIONING and Z-ORDER with a simpler, 
-- more adaptive approach. It automatically organizes data for fast 
-- queries based on the columns you cluster by.

-- Apply clustering to our Gold table — queries often filter by BU and region
ALTER TABLE gold_regional_supplier_coverage
CLUSTER BY (bu_code, region);

-- Apply to silver_agreements — queries often filter by status and type
ALTER TABLE silver_agreements
CLUSTER BY (document_type, agreement_status);

-- Verify clustering
DESCRIBE TABLE EXTENDED gold_regional_supplier_coverage;

In [0]:
%sql
-- ============================================================
-- PERFORMANCE: Efficient Joins and Reusing Intermediate Results
-- ============================================================
-- Bad pattern:  Running the same subquery multiple times
-- Good pattern: Use CTEs (WITH clause) or temp views to compute once, reuse many times

-- GOOD: CTE computes supplier_base once, then reused in two aggregations
WITH supplier_base AS (
  -- This is computed ONCE
  SELECT 
    supplier_number,
    supplier_name,
    bu_code,
    business_unit_name,
    region,
    supplier_type
  FROM silver_preferred_suppliers
),
by_region AS (
  SELECT region, COUNT(DISTINCT supplier_number) AS suppliers_in_region
  FROM supplier_base
  GROUP BY region
),
by_bu AS (
  SELECT bu_code, business_unit_name, COUNT(DISTINCT supplier_number) AS suppliers_in_bu
  FROM supplier_base
  GROUP BY bu_code, business_unit_name
)
SELECT 
  b.bu_code,
  b.business_unit_name,
  b.suppliers_in_bu,
  r.region,
  r.suppliers_in_region
FROM by_bu b
CROSS JOIN by_region r
ORDER BY b.bu_code, r.region;

In [0]:
from pyspark.sql import functions as F

# ============================================================
# PERFORMANCE: Broadcast Join for Small Tables
# ============================================================
# When joining a large table with a small lookup, use broadcast()
# to send the small table to all workers — avoids expensive shuffle.

# Our supplier lookup is small (~2,500 rows) — perfect for broadcast
preferred_suppliers = spark.table("silver_preferred_suppliers")
agreements = spark.table("silver_agreements")

print(f"Preferred suppliers: {preferred_suppliers.count():,} rows")
print(f"Agreements: {agreements.count():,} rows")

# Broadcast the smaller table for an efficient join
joined = agreements.join(
    F.broadcast(preferred_suppliers),  # <-- broadcast hint
    on="supplier_number",
    how="left"
)

# Explain the plan to see the BroadcastHashJoin
print("\n--- Query Plan (note BroadcastHashJoin) ---")
joined.explain(mode="simple")

## 💡 Performance Best Practices Summary

| Technique | Impact | When to use |
|-----------|--------|------------|
| **Liquid Clustering** | 2-10x faster queries | Tables queried with consistent filter columns |
| **Broadcast joins** | Eliminate shuffle | Small table (<\~1GB) joined to large table |
| **CTEs / Temp views** | Avoid recomputation | Same intermediate result used multiple times |
| **Column pruning** | Less I/O | `SELECT col1, col2` instead of `SELECT *` |
| **Predicate pushdown** | Scan less data | Filter early in the query (`WHERE` before `JOIN`) |
| **OPTIMIZE** | Compact small files | After many small writes (streaming, inserts) |

> **⚠️ Advanced exercise:** Run `EXPLAIN` on your queries to verify the optimizer is using 
> BroadcastHashJoin, filter pushdown, and column pruning.

---
# Section D — Building Pipelines with Notebooks & Lakeflow

> **Goal:** Structure multi-step data transformations into repeatable, monitored pipelines.
> We'll implement the **same Bronze → Silver → Gold logic** as a Lakeflow Spark Declarative Pipeline.

### Two Approaches to Pipelines

| Approach | Best for | Scheduling | Incremental? | Data Quality |
|----------|---------|-----------|-------------|-------------|
| **Notebook + Jobs** | Custom logic, ML, complex orchestration | Lakeflow Jobs triggers | Manual (you code it) | Manual checks |
| **Lakeflow Pipelines (SDP)** | Standard ETL/ELT, medallion architecture | Built-in scheduling | Automatic | Built-in expectations |

### What is Lakeflow Spark Declarative Pipelines (SDP)?
* You **declare** what your data should look like (schemas, quality rules)
* The framework handles **execution, ordering, retries, and incremental processing**
* Built-in **expectations** validate data quality at every step
* Automatic **lineage tracking** and **monitoring UI**

## D.1 — Structuring Notebook-Based Pipelines (Jobs Orchestration)

For the **notebook + Lakeflow Jobs** approach, you organize logic into separate notebooks
and orchestrate them as tasks in a Job:

```
enbridge_enablement/
├── pipelines/
│   ├── 01_ingest_bronze.py    ← Task 1: Ingest CSVs to Bronze
│   ├── 02_transform_silver.py ← Task 2: Clean & transform to Silver (depends on Task 1)
│   └── 03_aggregate_gold.py   ← Task 3: Aggregate to Gold (depends on Task 2)
└── Lesson_02_Building_Data_Pipelines.py  ← This notebook (workshop)
```

**Sections A and B above already contain the code for each step.** In production, you would
split those cells into separate notebooks and wire them as dependent tasks in a Lakeflow Job.

> **💡 Key benefit of Jobs:** You can mix Python, SQL, and even R notebooks in the same
> pipeline. You also get email/Slack notifications on success or failure.

## D.2 — Lakeflow Spark Declarative Pipeline (SDP)

Now let's implement the **same pipeline** as a Lakeflow pipeline. The code below shows
what the pipeline source files would contain. In a real setup, you'd create these as separate
files and attach them to a Lakeflow pipeline.

### Key SDP Concepts
* **Streaming Tables** — for Bronze (incremental ingestion with Auto Loader)
* **Materialized Views** — for Silver and Gold (auto-refresh on each pipeline run)
* **Expectations** — built-in data quality rules that warn, drop, or fail on violations
* **`STREAM`** keyword — reads data incrementally (only new rows since last run)

> ⚠️ **Note:** The SQL below uses Lakeflow SDP syntax. It runs **inside a pipeline**, not as
> standalone SQL. We show it here for learning — the actual pipeline would be created
> separately and linked to these source files.

In [0]:
%sql
-- ============================================================
-- LAKEFLOW SDP: Bronze Layer (Streaming Tables with Auto Loader)
-- ============================================================
-- This is SDP syntax — runs inside a Lakeflow pipeline, not standalone.
-- Copy this to a pipeline source file to use in production.

-- The full, working version lives in the repo at src/pipeline/01_bronze.sql (deployed by the bundle).
-- Line comments are used here because the glob '/*' in a path would open a nested /* block comment.

-- -- Bronze: Category Managed Suppliers (incremental ingestion)
-- CREATE OR REFRESH STREAMING TABLE bronze_category_managed
-- COMMENT 'Raw category-managed suppliers ingested incrementally from UC Volume'
-- AS SELECT
--   *,
--   _metadata.file_name AS source_file,
--   current_timestamp() AS ingested_at
-- FROM STREAM read_files(
--   '${volume_path}/*Category Managed*.csv',
--   format => 'csv',
--   header => 'true'
-- );
--
-- -- Bronze: Regional Sourcing Suppliers
-- CREATE OR REFRESH STREAMING TABLE bronze_regional_sourcing
-- COMMENT 'Raw regional sourcing suppliers ingested incrementally from UC Volume'
-- AS SELECT
--   *,
--   _metadata.file_name AS source_file,
--   current_timestamp() AS ingested_at
-- FROM STREAM read_files(
--   '${volume_path}/*Regional Sourcing*.csv',
--   format => 'csv',
--   header => 'true'
-- );
--
-- -- Bronze: Contracts (streaming read_files needs a directory or glob, not a single file)
-- CREATE OR REFRESH STREAMING TABLE bronze_contracts
-- COMMENT 'Raw legal contracts ingested incrementally from UC Volume'
-- TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
-- AS SELECT
--   *,
--   _metadata.file_name AS source_file,
--   current_timestamp() AS ingested_at
-- FROM STREAM read_files(
--   '${volume_path}/Contracts*.csv',
--   format => 'csv',
--   header => 'true'
-- );

SELECT 'SDP Bronze definitions shown above (uncomment inside a pipeline)' AS note;

In [0]:
%sql
-- ============================================================
-- LAKEFLOW SDP: Silver Layer (Materialized Views + Expectations)
-- ============================================================
-- Expectations are data quality rules that run on every pipeline update.
-- Three modes:
--   EXPECT     → warn only (log the violation, keep the row)
--   EXPECT ... ON VIOLATION DROP ROW     → filter out bad rows
--   EXPECT ... ON VIOLATION FAIL UPDATE  → stop the pipeline

/*
-- Silver: Unified Preferred Suppliers with quality checks
CREATE OR REFRESH MATERIALIZED VIEW silver_preferred_suppliers (
  -- EXPECTATIONS: Data quality rules
  CONSTRAINT valid_supplier_number EXPECT (supplier_number IS NOT NULL) 
    ON VIOLATION DROP ROW,
  CONSTRAINT valid_supplier_name EXPECT (supplier_name IS NOT NULL AND TRIM(supplier_name) != '') 
    ON VIOLATION DROP ROW,
  CONSTRAINT known_business_unit EXPECT (bu_code IN ('LP', 'GTM', 'GDS', 'GPT'))
    ON VIOLATION FAIL UPDATE
)
COMMENT 'Unified preferred suppliers with BU name resolution and data quality enforcement.'
AS
SELECT 
  `Supplier Name` AS supplier_name,
  CAST(`Supplier Number` AS INT) AS supplier_number,
  `Business Unit` AS bu_code,
  CASE `Business Unit`
    WHEN 'LP'  THEN 'Liquids Pipelines'
    WHEN 'GTM' THEN 'Gas Transmission'
    WHEN 'GDS' THEN 'Enbridge Gas'
    WHEN 'GPT' THEN 'Power Operations'
    ELSE `Business Unit`
  END AS business_unit_name,
  `Region` AS region,
  'Category Managed' AS supplier_type,
  _metadata.file_name AS source_file
FROM STREAM bronze_category_managed

UNION ALL

SELECT 
  `Supplier Name` AS supplier_name,
  CAST(`Supplier Number` AS INT) AS supplier_number,
  `Business Unit` AS bu_code,
  CASE `Business Unit`
    WHEN 'LP'  THEN 'Liquids Pipelines'
    WHEN 'GTM' THEN 'Gas Transmission'
    WHEN 'GDS' THEN 'Enbridge Gas'
    WHEN 'GPT' THEN 'Power Operations'
    ELSE `Business Unit`
  END AS business_unit_name,
  `Region` AS region,
  'Regional Sourcing' AS supplier_type,
  _metadata.file_name AS source_file
FROM STREAM bronze_regional_sourcing;
*/

SELECT 'SDP Silver definitions with expectations shown above' AS note;

In [0]:
%sql
-- ============================================================
-- LAKEFLOW SDP: Gold Layer (Materialized View - batch read)
-- ============================================================
-- Gold MVs read from Silver using batch (no STREAM keyword)
-- because they aggregate the full dataset.

/*
-- Gold: Regional Supplier Coverage
CREATE OR REFRESH MATERIALIZED VIEW gold_regional_supplier_coverage
COMMENT 'Regional supplier coverage for dashboards. Auto-refreshed by pipeline.'
AS
SELECT 
  bu_code,
  business_unit_name,
  region,
  supplier_type,
  COUNT(DISTINCT supplier_number) AS unique_suppliers,
  COUNT(*) AS total_assignments
FROM silver_preferred_suppliers
GROUP BY bu_code, business_unit_name, region, supplier_type;

-- Gold: Contract Analytics
CREATE OR REFRESH MATERIALIZED VIEW gold_contract_analytics (
  CONSTRAINT has_business_unit EXPECT (`business_unit` IS NOT NULL)
)
COMMENT 'Contract analytics by business unit. Gold layer.'
AS
SELECT 
  `Enbridge Business Unit` AS business_unit,
  `Contract Status` AS contract_status,
  `Contract Type` AS contract_type,
  COUNT(*) AS contract_count,
  COUNT(DISTINCT `Contract Party Name`) AS unique_suppliers
FROM bronze_contracts
WHERE `Enbridge Business Unit` IS NOT NULL
GROUP BY ALL;
*/

SELECT 'SDP Gold definitions shown above' AS note;

## D.3 — Data Quality with Lakeflow Expectations

Expectations are the built-in data validation feature of Lakeflow pipelines:

| Mode | Behavior | Use when |
|------|---------|----------|
| `EXPECT (condition)` | **Warn** — log violation, keep the row | Non-critical checks (e.g., optional fields) |
| `EXPECT ... ON VIOLATION DROP ROW` | **Quarantine** — drop failing rows | Data quality filters (e.g., NULL keys) |
| `EXPECT ... ON VIOLATION FAIL UPDATE` | **Fail** — stop the entire pipeline | Critical invariants (e.g., unknown BU code) |

### Monitoring Expectations
The Lakeflow pipeline UI shows:
* **Pass/fail counts** per expectation per run
* **Trend over time** — are quality issues increasing?
* **Which rows were dropped** (in the event log)

> **💡 Enbridge-specific example:** We set `known_business_unit` as `FAIL UPDATE` because
> an unknown BU code means the data source changed and needs human review.

---
# Section E — Monitoring, Re-runs & Troubleshooting

> **Goal:** Learn how to monitor pipeline health, set up alerts, handle failures,
> and support pipelines day-to-day.

### Key Operational Tasks
* Check pipeline run history and status
* Read logs to diagnose failures
* Set up email/Slack alerts on failure
* Re-run pipelines (full refresh vs. incremental)
* Common troubleshooting patterns

## E.1 — Monitoring Pipeline Runs

### For Notebook + Jobs Pipelines
1. **Lakeflow Jobs UI** → Click your job → see run history, duration, status
2. **Run details** → Click a run → see task-level logs, stdout/stderr
3. **API/CLI** → `databricks jobs list-runs --job-id <id>` for automation

### For Lakeflow Pipelines (SDP)
1. **Pipeline UI** → DAG view shows all datasets and their status
2. **Update history** → See every pipeline run with timing and row counts
3. **Event log** → Detailed events including expectation results
4. **Lineage** → Visual data flow from Bronze → Silver → Gold

In [0]:
%sql
-- ============================================================
-- MONITORING: Query Table History for Audit Trail
-- ============================================================
-- Every Delta table maintains a transaction log. Use DESCRIBE HISTORY
-- to see when tables were modified, by whom, and what operation.

DESCRIBE HISTORY silver_preferred_suppliers LIMIT 10;

In [0]:
%sql
-- ============================================================
-- MONITORING: Table Size and File Statistics
-- ============================================================
-- Useful for capacity planning and identifying tables that need OPTIMIZE

DESCRIBE DETAIL silver_preferred_suppliers;

## E.2 — Alerts, Failure Handling & Re-runs

### Setting Up Alerts
* **Lakeflow Jobs:** Configure email/webhook notifications in Job settings → "Notifications"
* **Lakeflow Pipelines:** Built-in email alerts for pipeline failures
* **Databricks SQL Alerts:** Create alerts on query results (e.g., "alert if data quality drops below 99%")

### Re-run Strategies

| Scenario | Action | Impact |
|----------|--------|--------|
| Pipeline failed mid-run | **Re-run** (incremental) | Picks up where it left off |
| Source data was corrected | **Re-run** (incremental) | Processes only new/changed data |
| Schema changed upstream | **Full refresh** | Reprocesses everything (expensive!) |
| Need to backfill historical data | **Full refresh** on specific tables | Targeted rebuild |

### Common Troubleshooting Patterns

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Pipeline stuck | Cluster starting or scaling | Wait; check cluster status |
| Schema mismatch error | Source file format changed | Check `_rescued_data` column; update schema |
| Expectation failure | Bad data in source | Review dropped rows; fix upstream |
| OOM (Out of Memory) | Join explosion or large shuffle | Use broadcast joins; increase cluster size |
| Duplicate rows | Missing deduplication logic | Add DISTINCT or window-based dedup |

---
# Bonus — Compute Selection for Performance & Cost

> **Quick touchpoint** (if time permits): Understanding compute options helps match
> the right resource to the right workload.

| Compute Type | Best for | Cost model | Key trait |
|-------------|---------|-----------|----------|
| **All-Purpose Cluster** | Interactive development, notebooks | Per-minute (while running) | Full Spark, customizable |
| **Jobs Cluster** | Scheduled pipelines, ETL | Per-minute (auto-terminates) | Cheaper than all-purpose |
| **SQL Warehouse** | SQL queries, dashboards, Genie | Per-query (serverless) | Fastest startup, SQL-only |
| **Serverless Compute** | Notebooks, Jobs (serverless) | Per-task | Zero config, fast startup |

### Matching Compute to Enbridge Workloads
* **Interactive analysis** (ad-hoc queries on Gold tables) → SQL Warehouse (serverless)
* **Scheduled ETL pipeline** (Bronze → Silver → Gold) → Jobs Cluster or Serverless
* **Lakeflow Pipeline** → Managed compute (auto-scaled by the pipeline)
* **Dashboard serving** → SQL Warehouse (serverless for best startup time)
* **ML model training** → All-Purpose Cluster with GPU (if needed)

---
# 🎯 Hands-On Exercise

Now it's your turn! Complete the following tasks:

### Exercise 1: Bronze Ingestion (10 min)
Using the pattern from Section A.2, ingest the `Agreements - BPA.csv` file with:
* A descriptive table comment
* `source_file` and `ingested_at` metadata columns
* Check the `_rescued_data` column for any schema issues

### Exercise 2: Silver Transformation (15 min)
Create a `silver_contracts_cleaned` table from `bronze_contracts` that:
* Applies Business Rule R4 (map BU code to full name)
* Filters to only `Active` contracts
* Adds a `contract_duration_days` column (`Contract End Date` - `Contract Start Date`)

### Exercise 3: Gold Aggregation (10 min)
Create a `gold_supplier_agreements` table that joins:
* `silver_preferred_suppliers` with `silver_agreements` on `supplier_number`
* Aggregates: count of agreements per supplier per BU
* Orders by total agreements descending

### 🌟 Stretch: Write an Expectation (5 min)
Convert one of your tables to SDP syntax with an expectation that:
* Drops rows where `supplier_number IS NULL`
* Warns (but keeps) rows where `contract_duration_days < 0`

---
# Lesson 2 — Summary

### What We Built

| Layer | Tables Created | Key techniques |
|-------|---------------|----------------|
| Bronze (original) | `bronze_agreements_bpa`, `bronze_agreements_cpa`, `bronze_contracts`, `bronze_category_managed`, `bronze_regional_sourcing` | `read_files()`, wildcard ingestion, rescued data, metadata columns |
| Bronze (enriched) | `bronze_supplier_headers`, `bronze_contract_mapping`, `bronze_po_line_items`, `bronze_supplier_responses`, `bronze_cost_centers`, `bronze_isn_region_mapping`, `bronze_region_mapping`, `bronze_cm_supplier_lists` | `read_files()` for CSV and Excel, wildcards across multiple files, column renaming |
| Silver (original) | `silver_preferred_suppliers`, `silver_agreements` | Business rules R1-R5, UNION ALL, CTE, type casting, rescued data extraction |
| Silver (enriched) | `silver_supplier_enriched`, `silver_contract_mapping`, `silver_po_spend` | LEFT JOIN dimension enrichment, R4 applied, amount casting |
| Gold (original) | `gold_regional_supplier_coverage`, `gold_agreement_summary`, `gold_contract_analytics` | Aggregations, clustering, tags, column comments |
| Gold (enriched) | `gold_supplier_spend_summary`, `gold_category_coverage`, `gold_supplier_compliance` | Spend aggregation from 50K PO lines, L3 category coverage, compliance status |
| Views | `vw_active_contracts` | Logical views for reusable query patterns |

### Key Takeaways
* **Unity Catalog** provides 3-level namespace, governance, and data quality enforcement
* **Medallion architecture** (Bronze -> Silver -> Gold) separates raw ingestion from business logic
* **Extending pipelines** with new datasets is additive -- new tables sit alongside existing ones
* **Excel ingestion** uses the same `read_files()` with `format => 'excel'` -- rename columns to snake_case
* **Lakeflow Pipelines** automate the orchestration with built-in expectations and monitoring
* **Performance techniques** (clustering, broadcast joins, CTEs) reduce cost and improve speed
* **Monitoring** through table history, pipeline UI, and alerts keeps pipelines healthy

### Table Count
* **13 Bronze tables** (5 original + 8 enriched)
* **5 Silver tables** (2 original + 3 enriched)
* **6 Gold tables** (3 original + 3 enriched)
* **1 View**

### Next: Lesson 3 -- Enabling Self-Service Analytics & AI for Business Users